In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import gc
import jax
gc.collect()
jax.clear_caches()

import superstats as sup
import numpy as np
from numba import njit, prange

## Simulator

In [ ]:
@njit(fastmath=True)
def sample_ddm_trial(
    v: float,
    a: float,
    bias: float,
    tau: float,
    dt: float = 0.001,
    s: float = 1.0,
    max_iter: int = 100000
) -> np.ndarray:
    n_iter = 0
    x = a * bias
    c = np.sqrt(dt * s)
    while x > 0 and x < a and n_iter < max_iter:
        x += v*dt + c * np.random.randn()
        n_iter += 1
    rt = n_iter * dt + tau
    resp = 0 if x <= 0 else 1
    return np.array([rt, resp], dtype=np.float32)


@njit(parallel=True, fastmath=True)
def sample_dynamic_ddm(
    v0_1: np.ndarray,
    v0_2: np.ndarray,
    v0_3: np.ndarray,
    v0_4: np.ndarray,
    dv_1: np.ndarray,
    dv_2: np.ndarray,
    dv_3: np.ndarray,
    dv_4: np.ndarray,
    a: np.ndarray,
    tau: np.ndarray,
    stim: np.ndarray,
    validity: np.ndarray,
    correct_resp: np.ndarray,
    dt: float = 0.001,
    s: float = 1.0,
    max_iter: int = 100000
) -> np.ndarray:
    num_steps = v0_1.shape[0]
    response_time = np.empty(num_steps, dtype=np.float32)
    choice = np.empty(num_steps, dtype=np.float32)
    noise_scale = s * np.sqrt(dt)

    v_intercepts


    stim = context[:, :, 0]
    num_iv = context[:, :, 1]
    validity = context[:, :, 2]
    correct_resp = context[:, :, 3]
    data = np.zeros((batch_size, num_obs, 2), dtype=np.float32)
    for b in prange(batch_size):
        for t in range(num_obs):
            sign = 1.0 if correct_resp[b, t] == 1 else -1.0
            current_bv = theta[b, t, int(stim[b, t])]
            drift = sign * (current_bv * validity[b, t])
            tau = theta[b, t, 5] + gamma[b, 0] * num_iv[b, t]
            data[b, t] = sample_ddm_trial(
                drift, theta[b, t, 4], gamma[b, 1], tau,
                dt=dt, s=s, max_iter=max_iter
            )
    return data